# Proyecto Loti Perú — Pipeline MLOps
> Notebooks listos para Databricks. Ajustá la variable `DATA_PATH` para tu ruta.

## 03 · Modelo (MLflow + Registro)

In [0]:
import mlflow, mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

CATALOG = "main"
SCHEMA  = "loterias_silver"
TABLE_FEATURES = f"{CATALOG}.{SCHEMA}.features_apuestas"
REGISTERED_NAME = "loti_fraude_logit"

# Cargar a Pandas
pdf = spark.table(TABLE_FEATURES).toPandas()

features = ["monto_log","freq_usuario","urgencia","repeticion_ip",
            "prom_monto_usuario","apuestas_7d_usuario","tasa_riesgo_ip"]
X, y = pdf[features], pdf["es_fraude"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

mlflow.set_experiment("/Shared/loti-mlops")  # ajusta tu ruta de experimento
with mlflow.start_run(run_name="logit_baseline_v1"):
    model = LogisticRegression(max_iter=2000)
    model.fit(Xtr, ytr)
    y_prob = model.predict_proba(Xte)[:,1]
    auc = roc_auc_score(yte, y_prob)
    mlflow.log_metric("roc_auc", float(auc))
    mlflow.log_params({"features": ",".join(features)})
    mlflow.sklearn.log_model(model, artifact_path="model", registered_model_name=REGISTERED_NAME)

print("ROC-AUC:", auc)
print(classification_report(yte, (y_prob>0.5).astype(int)))
print(f"Modelo registrado como '{REGISTERED_NAME}'")